# RNN을 이용한 텍스트 생성
- 문맥을 반영해 다음 단어를 예측하여 텍스트 생성하기 (다항 분류 - Many 2 One)

In [5]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences, to_categorical
from tensorflow.keras.layers import LSTM, Embedding, Dense
from tensorflow.keras.models import Sequential
import numpy as np

# """경마장에 있는 말이 뛰고 있다
# 그의 말이 법이다
# 가는 말이 고와야 오는 말이 곱다"""
text = \
'''17일은 아침부터 강한 햇볕이 내리쬐다가 오후 들어 갑작스러운 소나기가 쏟아지는 '변덕스러운 여름 날씨'가 이어지겠다.
한낮 자외선이 매우 강한 가운데 내륙 곳곳에는 천둥과 우박을 동반한 소나기까지 예보돼 외출 시 우산과 양산을 겸한 '우양산'을 챙기는 것이 좋겠다.
16일 기상청에 따르면 17일 우리나라는 서해상에 위치한 고기압 가장자리에 들겠다.
대기 하층에는 따뜻한 공기가 머무는 반면 상층에는 찬 공기가 유입되면서 대기 불안정이 커져 소나기구름이 발달할 전망이다.
소나기는 오전부터 18일 새벽 사이 수도권과 강원 내륙·산지, 충청 내륙, 호남, 경상권 등 대부분 내륙 지역에서 내리겠다. 예상 강수량은 5~40㎜다.
특히 소나기가 내리는 지역에서는 돌풍과 함께 천둥·번개가 치겠고 대기가 크게 불안정한 곳은 우박이 떨어질 가능성도 있다.
짧은 시간 강하게 비가 내리면서 도로가 미끄러워지고 가시거리가 급격히 짧아질 수 있어 운전자들의 주의가 요구된다.
비 소식에도 더위의 기세는 꺾이지 않는다.
17일 아침 최저기온은 16~22도, 낮 최고기온은 24~31도로 예보됐다.
주요 도시 예상 기온은 서울 22~31도, 인천 21~29도, 대전·대구 20~31도, 광주 21~30도, 부산 21~27도다.
전국 대부분 지역의 최고 체감온도는 31도 안팎까지 올라 무덥겠고, 밤사이 기온이 크게 떨어지지 않아 일부 지역에서는 열대야에 가까운 더위를 보이는 곳도 있겠다.
한낮 자외선도 강하겠다. 정오부터 오후 3시 사이 자외선 지수는 대부분 지역에서 '매우 높음' 또는 '높음' 수준을 나타낼 전망이다.
'매우 높음' 단계는 햇볕에 수십 분만 노출돼도 피부 화상을 입을 수 있는 수준이다.
수도권과 강원 영서, 충남 지역은 오존 농도도 '나쁨' 수준까지 치솟을 것으로 예상된다.
기상청은 야외 활동 시 충분한 수분을 섭취하고, 자외선 차단제 사용과 함께 우양산을 활용해 햇볕과 소나기에 모두 대비할 것을 당부했다.'''

# tok = Tokenizer(char_level=True) # 글자 단위
# tok = Tokenizer(char_level=False) # 단어 단위 , 기본값
tok = Tokenizer()
tok.fit_on_texts([text])
encoded = tok.texts_to_sequences([text])[0]
print(encoded)
print(tok.word_index)

vocab_size = len(tok.word_index) + 1 # 실제 단어 집합보다 + 1을 함 : keras의 토큰화 정수 인덱스는 1부터 출발, ont_hot인코딩은 0부터 시작
print(vocab_size)

# 훈련 데이터 작성
sequences = list()
for line in text.split('\n'): # 문장 토큰화
  enco = tok.texts_to_sequences([line])[0] # 행단위로 잘라서 라인단위로 인코딩
  # print(enco)
  # 바로 다음 단어를 label로 사용하기 위해 리스트에 담기
  for i in range(1, len(enco)):
    sequ = enco[:i+1]
    # print(sequ)
    sequences.append(sequ)
print('학습에 참여할 샘플 수 :',len(sequences))
print(sequences)
print(max(len(i) for i in sequences))

# 전체 각각의 벡터 길이를 통일
max_len = max(len(i) for i in sequences)
padding_sequence = pad_sequences(sequences, maxlen=max_len, padding='pre')
print(padding_sequence)
print()

# 각 벡터의 마지막 요소를 label로 사용 하기 위해 분리 작업 진행
x = padding_sequence[:, :-1] # feature
y = padding_sequence[:, -1] # label

print(x) # [[ 0  0  0  0  2] ...
print(y) # [ 3  1  4  5  1  7  1  9 10  1 11]

# label은 one-hot encoding 필요
y = to_categorical(y, num_classes=vocab_size)
print(y[:3]) # [[0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]

[26, 27, 5, 28, 29, 6, 30, 31, 7, 32, 33, 34, 35, 36, 8, 37, 38, 5, 39, 1, 40, 41, 42, 43, 44, 45, 46, 9, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 10, 57, 58, 59, 60, 61, 62, 11, 63, 64, 12, 65, 66, 67, 68, 12, 69, 11, 70, 71, 72, 73, 13, 74, 75, 76, 77, 14, 15, 16, 78, 79, 1, 80, 81, 82, 2, 1, 17, 83, 18, 84, 85, 86, 87, 7, 88, 19, 89, 20, 90, 91, 92, 21, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 22, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 10, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 18, 130, 131, 132, 3, 133, 4, 134, 135, 136, 3, 137, 4, 138, 139, 4, 140, 141, 2, 142, 143, 144, 3, 145, 146, 147, 148, 149, 21, 150, 151, 152, 19, 153, 154, 155, 156, 157, 158, 8, 159, 160, 161, 6, 162, 14, 23, 163, 2, 17, 24, 25, 164, 165, 166, 167, 13, 24, 25, 168, 169, 170, 171, 172, 173, 174, 175, 22, 176, 177, 15, 16, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 9, 191, 192, 193, 23, 194, 195, 20, 196, 197, 198, 199, 200, 201,

In [6]:
# model
model = Sequential()
model.add(Embedding(vocab_size, 32))
model.add(LSTM(32, activation='tanh'))
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(vocab_size, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model.summary())

model.fit(x, y, epochs=200, verbose=2)
print(model.evaluate(x, y))

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/200
7/7 - 2s - 332ms/step - accuracy: 0.0000e+00 - loss: 5.3195
Epoch 2/200
7/7 - 0s - 11ms/step - accuracy: 0.0138 - loss: 5.3170
Epoch 3/200
7/7 - 0s - 10ms/step - accuracy: 0.0184 - loss: 5.3155
Epoch 4/200
7/7 - 0s - 11ms/step - accuracy: 0.0276 - loss: 5.3132
Epoch 5/200
7/7 - 0s - 20ms/step - accuracy: 0.0276 - loss: 5.3100
Epoch 6/200
7/7 - 0s - 10ms/step - accuracy: 0.0276 - loss: 5.3053
Epoch 7/200
7/7 - 0s - 11ms/step - accuracy: 0.0184 - loss: 5.2965
Epoch 8/200
7/7 - 0s - 12ms/step - accuracy: 0.0184 - loss: 5.2764
Epoch 9/200
7/7 - 0s - 11ms/step - accuracy: 0.0230 - loss: 5.2400
Epoch 10/200
7/7 - 0s - 10ms/step - accuracy: 0.0230 - loss: 5.1931
Epoch 11/200
7/7 - 0s - 10ms/step - accuracy: 0.0230 - loss: 5.1338
Epoch 12/200
7/7 - 0s - 26ms/step - accuracy: 0.0276 - loss: 5.0571
Epoch 13/200
7/7 - 0s - 15ms/step - accuracy: 0.0323 - loss: 4.9808
Epoch 14/200
7/7 - 0s - 11ms/step - accuracy: 0.0323 - loss: 4.9162
Epoch 15/200
7/7 - 0s - 21ms/step - accuracy: 0

In [8]:
from IPython.lib.security import encode
# 문자열을 생성하는 predict  - 문자열 생성 함수 만들기
def sequences_gen_text(model, t, current_word, n):
  init_word = current_word # 처음 들어온 단어 저장
  sentence = ''
  for _ in range(n):
    encoded = t.texts_to_sequences([current_word])[0]
    encoded = pad_sequences([encoded], maxlen=max_len - 1, padding='pre')
    result = np.argmax(model.predict(encoded, verbose=0), axis=-1)

    # 예측 단어 찾기
    for word, index in t.word_index.items():
      # print(word, index)

      if index == result:
        break # 해당단어를 예측해서 for문을 나옴

    current_word = current_word + ' ' + word
    sentence = sentence + ' ' + word

  sentence = init_word + sentence
  return sentence

# print(sequences_gen_text(model, tok, '경마장', 5))
# print(sequences_gen_text(model, tok, '그의', 5))
# print(sequences_gen_text(model, tok, '고와야', 5))
print(sequences_gen_text(model, tok, '비가', 20))
print(sequences_gen_text(model, tok, '전국', 20))
print(sequences_gen_text(model, tok, '열대야', 20))

비가 새벽 사이 수도권과 강원 내륙·산지 충청 내륙 호남 경상권 등 대부분 내륙 지역에서 내리겠다 예상 강수량은 5 40㎜다 40㎜다 '우양산'을
전국 대부분 지역의 최고 체감온도는 31도 안팎까지 올라 무덥겠고 밤사이 기온이 크게 떨어지지 않아 일부 지역에서는 열대야에 가까운 더위를 보이는 곳도
열대야 아침 시간 강하게 비가 내리면서 도로가 미끄러워지고 가시거리가 급격히 짧아질 수 있어 운전자들의 주의가 요구된다 요구된다 높음' '높음' 수준을 나타낼
